# Intermediate Project – Smart Form Filler

## Objective

The objective of this project is to develop an AI-powered Smart Form Filler using LangChain, Groq, and Pydantic.

The system extracts structured information from natural language input and converts it into a validated form.

If required information is missing, the assistant asks clarification questions before producing the final structured output.

## Technologies Used

- Python
- LangChain
- Groq
- Pydantic
- Jupyter Notebook

# Project Description

Many users describe information in natural language rather than filling forms manually.

For example:

"I am looking for a remote Python developer job in Lahore with a salary around 250,000 PKR."

Instead of manually filling a form, the AI extracts the required information automatically and validates the result.

This project demonstrates structured output generation and intelligent validation using Large Language Models.

# Project Workflow

```

```
User Input

      │
      ▼

Prompt

      │
      ▼

Large Language Model

      │
      ▼

Structured Output

      │
      ▼

Validation

      │
      ▼

Missing Fields?

      │
      ├────────► Yes → Ask Clarification
      │
      ▼

Validated Form
```


In [1]:
from dotenv import load_dotenv

import os

from langchain_groq import ChatGroq

from pydantic import BaseModel, Field

load_dotenv()

True

In [2]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLM Initialized Successfully!")

Groq LLM Initialized Successfully!


# Creating the Form Schema

Pydantic is used to define the structure of the form.

Each extracted value must match the schema before it is considered valid.

In [3]:
from typing import Optional

class JobApplication(BaseModel):

    job_title: Optional[str] = Field(
        default=None,
        description="Job title"
    )

    company: Optional[str] = Field(
        default=None,
        description="Company name"
    )

    location: Optional[str] = Field(
        default=None,
        description="Job location"
    )

    salary_range: Optional[str] = Field(
        default=None,
        description="Salary offered"
    )

    required_skills: list[str] = Field(
        default_factory=list,
        description="Skills required"
    )

    remote: Optional[bool] = Field(
        default=None,
        description="Whether the job is remote"
    )

In [4]:
structured_llm = llm.with_structured_output(JobApplication)

print("Structured Output Ready!")

Structured Output Ready!


# Creating the Extraction Function

The AI model receives a natural language description from the user and extracts structured information according to the predefined Pydantic schema.

If some fields are not explicitly mentioned, the model should return `null` where appropriate instead of guessing.

In [5]:
def extract_job_details(user_input: str):

    prompt = f"""
You are an AI assistant that extracts job application information.

Extract the following fields from the user's input:

- job_title
- company
- location
- salary_range
- required_skills
- remote

Rules:

1. Do not guess missing information.
2. If information is unavailable, return null.
3. required_skills should always be a list.
4. remote should be true or false.

User Input:

{user_input}
"""

    return structured_llm.invoke(prompt)

# Validation Function

After extracting the information, we validate whether all important fields are available.

If required fields are missing, the assistant identifies them so that clarification questions can be asked.

In [6]:
def validate_form(job: JobApplication):

    missing = []

    if not job.job_title:
        missing.append("job_title")

    if not job.location:
        missing.append("location")

    if not job.required_skills:
        missing.append("required_skills")

    return missing

# Clarification Question Generator

When required information is missing, the assistant generates user-friendly clarification questions.

In [7]:
def ask_clarification(missing_fields):

    questions = {
        "job_title": "What is the job title?",
        "location": "Which city or location is this job in?",
        "required_skills": "What skills are required for this job?",
        "company": "What is the company name?",
        "salary_range": "What is the expected salary?",
        "remote": "Is this a remote job?"
    }

    for field in missing_fields:
        print("❓", questions[field])

# Test Case 1

The following example contains complete job information.

In [8]:
job = extract_job_details(
    """
Looking for a Remote Python Developer job at Google in Lahore.

Salary around 250000 PKR.

Skills required are Python, FastAPI and SQL.
"""
)

job

JobApplication(job_title='Python Developer', company='Google', location='Lahore', salary_range='250000 PKR', required_skills=['Python', 'FastAPI', 'SQL'], remote=True)

# Validation Example

In [9]:
missing = validate_form(job)

if missing:
    ask_clarification(missing)
else:
    print("✅ Form is complete.")

✅ Form is complete.


# Test Case 2 – Missing Information

This example intentionally omits some fields to demonstrate validation and clarification.

In [10]:
job = extract_job_details(
    """
Looking for a Data Analyst position.

I know Python and Excel.
"""
)

job

JobApplication(job_title='Data Analyst', company=None, location=None, salary_range=None, required_skills=['Python', 'Excel'], remote=None)

In [11]:
missing = validate_form(job)

if missing:
    ask_clarification(missing)
else:
    print("✅ Form is complete.")

❓ Which city or location is this job in?


# Test Case 3 – Remote Software Engineer

This example contains complete information for a remote software engineering position.

In [12]:
job = extract_job_details(
    """
Hiring a Remote Software Engineer at Microsoft in Islamabad.

Salary is around 400000 PKR.

Required skills are Python, Docker, Kubernetes, and AWS.
"""
)

print(job)

missing = validate_form(job)

if missing:
    ask_clarification(missing)
else:
    print("✅ Form is complete.")

job_title='Software Engineer' company='Microsoft' location='Islamabad' salary_range='400000 PKR' required_skills=['Python', 'Docker', 'Kubernetes', 'AWS'] remote=True
✅ Form is complete.


# Test Case 4 – Incomplete Job Description

This example intentionally leaves out multiple fields to test the validation process.

In [13]:
job = extract_job_details(
    """
Need a Frontend Developer.

Skills: HTML, CSS, JavaScript.
"""
)

print(job)

missing = validate_form(job)

if missing:
    ask_clarification(missing)
else:
    print("✅ Form is complete.")

job_title='Frontend Developer' company=None location=None salary_range=None required_skills=['HTML', 'CSS', 'JavaScript'] remote=None
❓ Which city or location is this job in?


# Test Case 5 – AI Engineer Position

This example demonstrates another complete job description.

In [14]:
job = extract_job_details(
    """
Hiring an AI Engineer at OpenAI in San Francisco.

Salary: 180000 USD.

Skills required:
Python, PyTorch, Machine Learning, LangChain.

Remote work available.
"""
)

print(job)

missing = validate_form(job)

if missing:
    ask_clarification(missing)
else:
    print("✅ Form is complete.")

job_title='AI Engineer' company='OpenAI' location='San Francisco' salary_range='180000 USD' required_skills=['Python', 'PyTorch', 'Machine Learning', 'LangChain'] remote=True
✅ Form is complete.


# Validation Summary

The Smart Form Filler successfully extracts structured information from natural language input.

When required information is missing, the validation process identifies the missing fields and generates clarification questions instead of guessing the values.

# Observations

During this project, the following observations were made:

- Natural language input can be converted into structured data using LLMs.
- Pydantic ensures that extracted information follows a predefined schema.
- Validation prevents incomplete forms from being accepted.
- Clarification questions improve the quality of collected information.
- Structured outputs are highly useful for automation workflows.

# Conclusion

In this project, an AI-powered Smart Form Filler was developed using LangChain, Groq, and Pydantic.

The system automatically extracted structured job information from natural language descriptions, validated the extracted data, and identified missing information through clarification questions.

This project demonstrates how Large Language Models can simplify data collection and automate form-filling tasks while maintaining data quality through structured validation.